In [ ]:
# ========================================================
# 03_autoencoder.ipynb
# Autoencoder po rozszerzeniu zbioru normalnego
# ========================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import joblib
import os

# ========================================================
# Ustawienia
# ========================================================

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 1024
EPOCHS = 30
LEARNING_RATE = 0.001
THRESHOLD_QUANTILE = 0.95

print("=== FINALNY MODEL AUTOENCODER - PO ROZSZERZENIU NORMAL DATASET ===")
print("Urządzenie:", DEVICE)

# ========================================================
# 1. Ładowanie danych
# ========================================================

normal_df = pd.read_csv("../data/processed/normal_features.csv")
feature_columns = normal_df.columns.tolist()

print(f"Liczba cech: {len(feature_columns)}")
print(f"Liczba flowów normalnych: {len(normal_df)}")

X_normal = torch.tensor(normal_df.values, dtype=torch.float32)

dataset = TensorDataset(X_normal)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ========================================================
# 2. Model
# ========================================================

class ImprovedAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = ImprovedAutoencoder(len(feature_columns)).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion = nn.MSELoss()

# ========================================================
# 3. Trening
# ========================================================

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for (batch,) in loader:
        batch = batch.to(DEVICE)

        recon = model(batch)
        loss = criterion(recon, batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.size(0)

    avg_loss = total_loss / len(dataset)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.6f}")

model.eval()

# ========================================================
# 4. Funkcja liczenia błędu rekonstrukcji batchami
# ========================================================

def compute_errors(df, batch_size=4096):
    X = torch.tensor(df.values, dtype=torch.float32)
    dl = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=False)

    all_errors = []

    model.eval()
    with torch.no_grad():
        for (batch,) in dl:
            batch = batch.to(DEVICE)
            recon = model(batch)

            # max error po cechach
            errors = torch.max((batch - recon) ** 2, dim=1).values
            all_errors.append(errors.cpu())

    return torch.cat(all_errors).numpy()

# ========================================================
# 5. Błędy dla normalnego ruchu i próg
# ========================================================

normal_errors = compute_errors(normal_df)

threshold = np.quantile(normal_errors, THRESHOLD_QUANTILE)

print("\n=== Statystyki błędu rekonstrukcji dla normal ===")
print("Mean:", round(float(np.mean(normal_errors)), 6))
print("Median:", round(float(np.median(normal_errors)), 6))
print("Max:", round(float(np.max(normal_errors)), 6))
print(f"Threshold q={THRESHOLD_QUANTILE}:", round(float(threshold), 6))

# ========================================================
# 6. Ewaluacja scenariuszy
# ========================================================

scenarios = [
    "guloader",
    "scanning",
    "njrat",
    "kongtuke1",
    "kongtuke2",
    "remcos",
    "xloader",
    "xworm",
    "phantomstealer"
]

results = []

# normal jako FPR
detected_normal = int((normal_errors > threshold).sum())
results.append({
    "scenario": "normal (FPR)",
    "AE_mean_error": float(np.mean(normal_errors)),
    "AE_median_error": float(np.median(normal_errors)),
    "AE_max_error": float(np.max(normal_errors)),
    "AE_detected": detected_normal,
    "AE_total": len(normal_errors),
    "Autoencoder DR (%)": round(detected_normal / len(normal_errors) * 100, 2)
})

for sc in scenarios:
    df = pd.read_csv(f"../data/processed/{sc}_features.csv")
    df = df.reindex(columns=feature_columns, fill_value=0)

    errors = compute_errors(df)

    detected = int((errors > threshold).sum())
    total = len(errors)

    results.append({
        "scenario": sc,
        "AE_mean_error": float(np.mean(errors)),
        "AE_median_error": float(np.median(errors)),
        "AE_max_error": float(np.max(errors)),
        "AE_detected": detected,
        "AE_total": total,
        "Autoencoder DR (%)": round(detected / total * 100, 2)
    })

df_results = pd.DataFrame(results)

print("\n=== WYNIKI AUTOENCODERA ===")
print(df_results[["scenario", "AE_detected", "AE_total", "Autoencoder DR (%)"]])

os.makedirs("../results", exist_ok=True)
df_results.to_csv("../results/final_ae_results.csv", index=False)

print("\nZapisano: ../results/final_ae_results.csv")

# ========================================================
# 7. AUC-ROC i AUC-PR
# ========================================================

all_scores = []
all_labels = []

all_scores.append(normal_errors)
all_labels.append(np.zeros(len(normal_errors)))

for sc in scenarios:
    df = pd.read_csv(f"../data/processed/{sc}_features.csv")
    df = df.reindex(columns=feature_columns, fill_value=0)

    errors = compute_errors(df)

    all_scores.append(errors)
    all_labels.append(np.ones(len(errors)))

y_scores = np.concatenate(all_scores)
y_true = np.concatenate(all_labels)

auc_roc = roc_auc_score(y_true, y_scores)
auc_pr = average_precision_score(y_true, y_scores)

print("\n=== METRYKI GLOBALNE ===")
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")

# Precision-Recall Curve
prec, rec, th = precision_recall_curve(y_true, y_scores)

plt.figure(figsize=(8, 6))
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Autoencoder")
plt.grid(True)
plt.tight_layout()
plt.show()

# ========================================================
# 8. Zapis modelu
# ========================================================

torch.save(model.state_dict(), "../models/best_autoencoder_final.pth")

print("\nModel zapisany jako: ../models/best_autoencoder_final.pth")
print("Encoder output shape:", model.encoder(torch.randn(1, len(feature_columns)).to(DEVICE)).shape)